# 00 해설 · 모델 파일과 학습 이미지는 어떻게 준비할까요?

[00번 실습 노트북](../notebooks/00_hf_download_and_data.ipynb)의 **코드 셀 17개**를 하나씩 풀어 읽는 안내서입니다. 처음부터 모든 문법을 외우기보다, 각 셀이 무엇을 받아 무엇을 남기는지 따라가 보세요.

**이 파일은 읽기용 해설 노트북입니다.** 원본 코드는 회색 코드 블록에 그대로 실었으며 실행되는 셀이 아닙니다. `Run All`을 눌러도 모델 다운로드·GPU 할당·파일 저장은 일어나지 않습니다. 실행할 수 있는 다섯 개의 작은 예제는 Python 기본 기능만 사용하며 CPU에서 동작합니다. 추가 라이브러리 설치도 필요 없습니다.

실제 모델 파일과 이미지를 준비할 때는 **원본 00번**을 실행하세요. 이 해설을 실행한 것만으로 원본 01·02번의 준비가 끝나지는 않습니다.

이 해설의 ‘원본 N번째 셀’은 제목·설명 셀도 포함해 위에서부터 센 위치입니다. 노트북 실행 횟수를 표시하는 `[1]`, `[2]`와는 다릅니다.

## 읽을 순서

| 단계 | 원본 셀 위치 | 남기는 것 |
|---|---|---|
| 작업할 폴더와 도구 확인 | 6·7·9번째 | 프로젝트 경로, 라이브러리, CLI 경로 |
| 모델 내려받기와 확인 | 11·12번째 | 모델 세 파일과 다운로드 기록 |
| 이미지 선택 조건과 검사 함수 | 15·17·18·20번째 | 클래스 목록, 선택 기준, 재사용 여부 |
| 데이터 내려받기와 이미지 선택 | 22·24·26·28번째 | 학습·검증·테스트 이미지와 정답 |
| 저장·다시 읽기·눈으로 확인 | 30·31·32·34번째 | `.npz` 세 파일과 `manifest.json`, 이미지 표 |

## 코드에서 자주 만날 기호

| 표현 | 읽는 방법 |
|---|---|
| `name = value` | 오른쪽 값을 왼쪽 이름에 연결합니다. 수학의 ‘같다’와 다릅니다. |
| `a == b` / `a != b` | 같은지 / 다른지 비교합니다. 결과는 `True` 또는 `False`입니다. |
| `[a, b]` / `{"name": value}` | 순서가 있는 목록 / 이름으로 값을 찾는 사전입니다. |
| `for item in items:` | 항목을 하나씩 꺼내 들여쓴 부분을 반복합니다. |
| `if 조건:` | 조건이 맞을 때만 들여쓴 부분을 실행합니다. |
| `def 함수(입력):` / `return 결과` | 다시 사용할 작업을 정의하고, 호출한 곳에 결과를 돌려줍니다. |
| `with ... as 이름:` | 자원을 사용하는 구간을 정합니다. 여기서는 구간이 끝나면 파일 등을 닫습니다. |

## Codespaces에서 00번을 실행한다는 뜻

저장소의 [devcontainer 설정](../../.devcontainer/devcontainer.json)이 수업 환경을 만들고 [설치 스크립트](../../scripts/setup.sh)가 필요한 라이브러리와 커널을 준비합니다. **커널**은 노트북의 Python 코드를 실제로 실행하는 프로그램입니다. 원본 00번에서는 `Vision AI (Codespaces CPU)`를 선택합니다.

`hf download`는 모델 파일을 받는 명령이고, Colab CLI는 이후 다른 컴퓨터의 GPU 세션을 연결하는 도구입니다. 00번을 실행하는 동안에는 Colab GPU를 만들지 않습니다. 자세한 환경 설정과 복구 순서는 [원본 00번 앞부분](../notebooks/00_hf_download_and_data.ipynb)을 참고하세요.

## 원본 6번째 셀 · 작업할 폴더 찾기

아래는 **읽기용 원본 코드**입니다.

```python
from pathlib import Path
import os
import sys
import json
import hashlib
from importlib.metadata import version

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((path for path in candidates if (path / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise RuntimeError(".vision-lab-root가 있는 수업 폴더에서 열거나 Colab에 실습 파일을 먼저 업로드하세요.")
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
print("Project:", ROOT)
print("Python:", sys.executable)
```

**입력 → 출력:** 노트북을 연 위치를 바탕으로 프로젝트 폴더 `ROOT`를 찾고, 그 경로와 현재 Python 경로를 출력합니다.

- `from pathlib import Path`는 경로를 다루는 도구 하나를 가져옵니다. `import os`처럼 쓰면 도구 모음 전체를 가져와 `os.chdir(...)`처럼 이름을 붙여 사용합니다.
- `sys`는 현재 Python 정보, `json`은 기록 파일의 읽기·쓰기, `hashlib`는 파일 내용의 지문 계산에 사용합니다. `version`은 설치한 라이브러리 버전을 확인하는 함수입니다. 모두 Python 기본 도구입니다.
- `Path.cwd()`는 지금 작업 중인 폴더입니다. `.parents`에는 그 폴더의 상위 폴더들이 들어 있습니다. 앞의 `*`는 여러 경로를 목록 안에 펼쳐 넣습니다.
- `candidates`에는 현재 폴더, 상위 폴더들, Colab에서 사용하는 예비 경로가 순서대로 들어갑니다. Colab 경로가 적혀 있다고 원격 연결이 생기지는 않습니다.
- `(path / ".vision-lab-root")`에서 `/`는 경로를 이어 붙입니다. 숫자를 나누는 계산이 아닙니다. `.is_file()`은 그 위치에 기준 파일이 있는지 확인합니다.
- `next((... for ... if ...), None)`는 조건에 맞는 경로 중 **첫 번째**를 고릅니다. 찾지 못하면 `None`, 즉 ‘값 없음’을 남깁니다.
- `ROOT is None`이면 `raise RuntimeError(...)`로 중단합니다. 이때 기준 파일을 임의로 만들기보다, 수업 저장소의 올바른 폴더를 열었는지 확인합니다.
- `os.chdir(ROOT)`는 이후의 상대 경로가 프로젝트 폴더에서 시작하게 합니다. `setdefault`는 그래프 설정 폴더 값을 **아직 설정하지 않았을 때만** 넣습니다.
- `print`는 확인할 내용을 화면에 보여줍니다. `Python:` 뒤가 엉뚱한 환경을 가리키면, 설치를 반복하기 전에 원본 노트북의 커널을 다시 확인하세요.

## 원본 7번째 셀 · 배열·이미지·그래프 도구 가져오기

아래는 **읽기용 원본 코드**입니다.

```python
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
for package in ("numpy", "Pillow", "matplotlib", "ipykernel"):
    print(f"{package}: {version(package)}")
```

**입력 → 출력:** 설치된 도구를 가져와 사용할 준비를 하고, 그래프 모양과 버전 출력을 설정합니다. 아직 이미지를 읽거나 학습하지 않습니다.

- `numpy as np`는 NumPy를 짧은 이름 `np`로 쓰겠다는 뜻입니다. NumPy의 **배열**은 이미지의 픽셀처럼 많은 숫자를 묶어 계산하는 자료형입니다.
- `from PIL import Image`는 Pillow의 이미지 도구를 가져옵니다. 설치 이름은 `Pillow`, Python에서 가져오는 이름은 `PIL`이므로 서로 달라도 정상입니다.
- `matplotlib.pyplot as plt`는 그래프를 그리는 기능을 `plt`라는 이름으로 사용합니다. `as`는 다른 이름을 붙이는 문법입니다.
- `get_ipython()`은 노트북 실행 환경을 확인합니다. `if ipython is not None`는 그런 환경이 있을 때만 다음 설정을 하게 합니다.
- `run_line_magic("matplotlib", "inline")`은 그래프를 노트북 출력 안에 보이게 합니다. 원본 셀이 일반 Python 파일에서 실행되는 경우에는 이 부분을 건너뜁니다.
- `plt.rcParams.update({...})`는 그래프 기본 모양을 한꺼번에 바꿉니다. `figure.figsize`는 가로·세로 크기, `font.size`는 글자 크기입니다.
- `axes.spines.top/right: False`는 그래프의 위쪽·오른쪽 테두리를 숨깁니다. 데이터나 학습 결과에는 영향을 주지 않습니다.
- `for package in (...)`는 네 패키지 이름을 차례로 꺼냅니다. `f"{package}: {version(package)}"`의 중괄호 안에는 실제 값이 들어갑니다.
- `ModuleNotFoundError`가 나오면 그 커널에서 도구를 찾지 못한 상태입니다. 원본의 환경 설치 안내와 커널 선택을 먼저 확인합니다.

## 원본 9번째 셀 · HF 명령과 모델 주소 정하기

아래는 **읽기용 원본 코드**입니다.

```python
import subprocess

HF = Path(sys.executable).parent / "hf"
if not HF.is_file():
    raise RuntimeError("현재 커널에 hf CLI가 없습니다. scripts/setup.sh 실행 후 커널을 다시 선택하세요.")
print("Hugging Face Hub:", version("huggingface_hub"))
print("CLI:", HF)
MODEL_ID = "facebook/deit-tiny-patch16-224"
MODEL_REVISION = "b3428f18dcc7b543470d07f14b4a4157815d1880"
MODEL_DIR = ROOT / "hf_colab_gpu/models/deit-tiny"
MODEL_FILES = ["config.json", "preprocessor_config.json", "pytorch_model.bin"]
```

**입력 → 출력:** 현재 Python과 같은 환경의 `hf` 실행 파일을 찾고, 모델 이름·버전·저장 위치·필요한 파일 이름을 변수에 담습니다.

- `subprocess`는 Python에서 별도의 프로그램을 실행할 때 씁니다. 다음 셀에서 터미널 명령인 `hf download`를 호출할 준비입니다.
- `sys.executable`은 지금 커널의 Python 실행 파일 경로입니다. `Path(...).parent / "hf"`는 **같은 폴더의 HF CLI**를 가리킵니다. 다른 가상환경의 명령과 뒤섞이지 않게 합니다.
- `if not HF.is_file()`의 `not`은 참·거짓을 뒤집습니다. 실행 파일이 없으면 다음 다운로드를 시도하지 않고 안내문과 함께 멈춥니다.
- `MODEL_ID`는 Hub에서 찾을 모델의 이름입니다. 여기서는 `facebook/deit-tiny-patch16-224`를 사용합니다.
- `MODEL_REVISION`은 특정 시점의 모델 파일을 가리키는 식별자입니다. 모델 이름만 같고 내용은 달라지는 일을 줄이려고 고정합니다.
- `MODEL_DIR`는 파일을 받을 폴더입니다. `ROOT / ...`처럼 경로를 조합하므로 학생마다 프로젝트 최상위 경로가 달라도 같은 하위 위치를 찾습니다.
- `MODEL_FILES`는 세 파일을 순서대로 담은 목록입니다. `config.json`은 모델 구조·라벨 설정, `preprocessor_config.json`은 이미지 전처리 설정, `pytorch_model.bin`은 학습된 가중치를 담습니다.
- 대문자 변수명은 ‘설정값’임을 알아보기 위한 관례입니다. Python이 값을 잠가 주는 것은 아닙니다.
- 모델을 바꾸려면 이름만 바꾸는 것으로 끝나지 않습니다. 필요한 파일, revision, 다음 셀의 파일 지문도 함께 맞춰야 합니다.

## 원본 11번째 셀 · Python에서 실제 다운로드 명령 실행하기

아래는 **읽기용 원본 코드**입니다.

```python
download_env = os.environ.copy()
download_env["HF_HOME"] = str(ROOT / ".cache/huggingface")
download_env["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
download_env["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
command = [str(HF), "download", MODEL_ID, *MODEL_FILES,
           "--revision", MODEL_REVISION, "--local-dir", str(MODEL_DIR)]
print(" ".join(command))
subprocess.run(command, check=True, env=download_env, timeout=600)
```

**입력 → 출력:** 앞에서 정한 모델 설정으로 HF CLI를 실행하고 `MODEL_DIR`에 모델 파일 세 개를 내려받습니다. 실행은 **원본 00번에서만** 일어납니다.

- `os.environ.copy()`는 프로그램 설정값인 환경변수를 복사합니다. 원본 설정을 직접 바꾸지 않고 다운로드 프로그램에 전달할 설정을 따로 만듭니다.
- `download_env["HF_HOME"] = ...`는 HF가 캐시를 사용할 위치를 정합니다. **캐시**는 다음 요청에서 다시 쓸 수 있도록 남겨 두는 파일 보관소입니다.
- `HF_HUB_DISABLE_IMPLICIT_TOKEN = "1"`은 저장된 로그인 토큰을 이 공개 다운로드에 자동으로 보내지 않도록 설정합니다. 이 코드에는 로그인이 필요한 비공개 모델을 받는 절차가 없습니다.
- `HF_HUB_DOWNLOAD_TIMEOUT = "120"`은 HF 다운로드 동작의 대기 설정이고, 맨 아래 `timeout=600`은 별도 프로그램 전체를 기다리는 최대 시간입니다. 둘은 같은 타이머가 아닙니다.
- `command`는 명령어와 인자를 **각각 별도 문자열로 담은 목록**입니다. `*MODEL_FILES`는 파일 세 이름을 그 목록에 하나씩 펼칩니다.
- `--revision` 뒤에는 모델 버전, `--local-dir` 뒤에는 저장 경로가 들어갑니다. `str(Path)`는 경로 객체를 외부 명령에 전달할 문자열로 바꿉니다.
- `" ".join(command)`는 목록 사이에 공백을 넣어 읽기 좋은 한 줄을 만들 뿐입니다. 화면에 명령을 출력하는 것 자체가 다운로드는 아닙니다.
- `subprocess.run(...)`이 실제 실행입니다. `check=True`이므로 프로그램이 실패하면 Python도 오류로 멈춥니다. `env=download_env`로 위에서 준비한 설정을 전달합니다.
- 연결 문제나 시간 초과가 나면 그 셀의 오류부터 확인하세요. 다운로드가 끝나기 전에 다음 파일 검사 셀을 실행하면 파일이 없다는 오류가 날 수 있습니다.

## 원본 12번째 셀 · 모델 파일이 같은 내용인지 검사하기

아래는 **읽기용 원본 코드**입니다.

```python
EXPECTED_MODEL_SHA256 = {'config.json': 'e8786a68c993c04b8721c4d379e4f9de6a9564fba5987704dbf8696d53b7972d', 'preprocessor_config.json': '90d1ae427c27d2b9dcb01f6a62ba96aa67aac2de1697c59cd212055b5035c4a2', 'pytorch_model.bin': 'e1a51b0c81ff812e079d2189352747947879fee56429e4480064a6695cb34b2c'}
downloaded_hashes = {}
for filename in MODEL_FILES:
    path = MODEL_DIR / filename
    downloaded_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
    if downloaded_hashes[filename] != EXPECTED_MODEL_SHA256[filename]:
        raise ValueError(f"모델 파일이 검증한 원본과 다릅니다: {filename}")
    print(filename, path.stat().st_size, "bytes · SHA256 확인")
download_manifest = {"model_id": MODEL_ID, "revision": MODEL_REVISION,
                     "files": downloaded_hashes, "download_method": "hf download"}
(MODEL_DIR / "download_manifest.json").write_text(
    json.dumps(download_manifest, indent=2), encoding="utf-8")
```

**입력 → 출력:** 내려받은 세 파일의 지문을 수업에서 확인한 값과 비교하고, 통과한 파일의 정보로 `download_manifest.json`을 만듭니다.

- `EXPECTED_MODEL_SHA256`은 ‘파일 이름 → 기대하는 지문’ 사전입니다. 긴 문자열을 외울 필요는 없고, 같은 파일인지 비교하는 기준이라고 이해하면 됩니다.
- `downloaded_hashes = {}`는 결과를 담을 빈 사전입니다. 반복문에서 파일 이름을 열쇠로 써 지문을 하나씩 넣습니다.
- `path.read_bytes()`는 파일 내용을 바이트로 읽습니다. **바이트**는 파일을 이루는 작은 숫자 단위입니다. SHA256은 그 내용으로 일정한 길이의 지문을 계산합니다.
- `.hexdigest()`는 계산 결과를 문자로 표현합니다. 이름이 같은 파일이라도 내용이 달라지면 이 지문을 비교해 차이를 발견할 수 있습니다.
- `!=`로 기대값과 다름을 확인하면 즉시 중단합니다. 오류를 없애려고 기대값을 새 파일 값으로 덮어쓰면 이 확인의 목적이 사라집니다.
- `path.stat().st_size`는 파일 크기를 바이트 단위로 보여줍니다. ‘SHA256 확인’ 문구는 해당 파일이 비교를 통과한 뒤에만 출력됩니다.
- `download_manifest`는 모델 이름, revision, 세 지문, 다운로드 방법을 묶은 **기록용 사전**입니다. `json.dumps(..., indent=2)`는 읽기 좋은 JSON 글자로 바꿉니다.
- `.write_text(..., encoding="utf-8")`는 그 글자를 파일로 저장합니다. 따라서 이 원본 셀은 검사뿐 아니라 기록 파일 생성도 수행합니다.
- 지문 일치는 ‘선택한 기준 파일과 내용이 같다’는 확인입니다. 모델의 정확도나 모든 보안 위험까지 보증하는 검사는 아닙니다.

### 작은 실행 예제 · 내용이 바뀌면 지문도 달라집니다

파일 대신 짧은 바이트를 사용해 같은 내용과 다른 내용을 비교합니다. `b"cup"`의 `b`는 글자를 바이트로 쓴다는 뜻입니다.

**설명용 가상 데이터입니다. 실제 모델·CIFAR 이미지나 GPU 결과를 계산하지 않습니다.**

In [1]:
import hashlib

first = hashlib.sha256(b"cup").hexdigest()
same = hashlib.sha256(b"cup").hexdigest()
changed = hashlib.sha256(b"Cup").hexdigest()
print("같은 내용의 지문이 같은가:", first == same)
print("한 글자를 바꾼 지문이 같은가:", first == changed)
print("지문의 문자 수:", len(first))

같은 내용의 지문이 같은가: True
한 글자를 바꾼 지문이 같은가: False
지문의 문자 수: 64


**확인:** 출력은 `True`, `False`, `64`입니다. 이 예제에서는 대문자 한 글자로 바꿔도 지문이 달라집니다. 파일 크기나 모델 점수와는 관계없는 값입니다.

## 원본 15번째 셀 · 다섯 종류의 물체와 분할 개수 정하기

아래는 **읽기용 원본 코드**입니다.

```python
import io
import shutil
import urllib.request
import pyarrow.parquet as pq

print("pyarrow:", version("pyarrow"))
CLASSES = ["bottle", "bowl", "can", "cup", "plate"]
FINE_LABEL_IDS = [9, 10, 16, 28, 61]
DATASET_ID = "uoft-cs/cifar100"
DATASET_REVISION = "aadb3af77e9048adbea6b47c21a81e47dd092ae5"
SOURCE_FILES = {
    "train": "694865d6b990e234804f01268586c41e88bcbbb75e20858432c05ad4081aca23",
    "test": "98776c529bb146a9c791229df74a5cf076be9b43d82dbbd334b6a7788d73dc68",
}
SEED = 42
TRAIN_PER_CLASS, VALIDATION_PER_CLASS, TEST_PER_CLASS = 100, 20, 40
CACHE = ROOT / ".cache/data"
DATA_PATH = ROOT / "data/prepared"
print("Classes:", dict(zip(FINE_LABEL_IDS, CLASSES)))
```

**입력 → 출력:** 공개 데이터셋의 클래스 번호, 데이터 버전, 파일 지문, 선택할 이미지 수와 저장 위치를 설정합니다.

- `io.BytesIO`는 나중에 메모리 안의 이미지 바이트를 파일처럼 읽게 합니다. `shutil`은 다운로드 내용을 복사하고 `urllib.request`는 인터넷 주소에 요청을 보냅니다.
- `pyarrow.parquet as pq`는 **Parquet**라는 표 저장 형식을 읽습니다. 표의 한 행에는 이미지와 정답 번호가 들어 있습니다. 이 도구는 별도로 설치된 라이브러리입니다.
- `CLASSES`에는 `bottle`, `bowl`, `can`, `cup`, `plate`가 순서대로 들어갑니다. 이 순서는 뒤에서 새 라벨 번호 `0~4`와 연결됩니다.
- `FINE_LABEL_IDS`는 원본 CIFAR-100에서 이 물체들의 번호입니다. 예를 들어 원본 `28`이 `cup`이며, 수업용으로 바꾸면 네 번째인 `3`이 됩니다.
- `DATASET_ID`와 `DATASET_REVISION`은 이미지 데이터를 받을 위치와 시점을 정합니다. `SOURCE_FILES`는 원본 학습·테스트 파일의 기대 지문입니다.
- `SEED = 42`는 무작위 선택을 다시 같은 순서로 재현하기 위한 시작값입니다. ‘42가 좋은 이미지를 고른다’는 뜻은 아닙니다.
- `TRAIN_PER_CLASS, ... = 100, 20, 40`은 세 변수에 값을 한 번에 넣습니다. **클래스마다** 이 개수이므로 다섯 클래스를 합치면 학습 500장, 검증 100장, 테스트 200장입니다.
- `CACHE`는 원본 파일 보관 장소, `DATA_PATH`는 수업에 쓸 이미지만 추려 저장할 장소입니다. 둘의 역할이 다릅니다.
- `dict(zip(FINE_LABEL_IDS, CLASSES))`는 같은 위치의 번호와 이름을 짝지어 사전으로 만들고, 잘 연결됐는지 출력합니다. 두 목록의 순서를 따로 바꾸지 않도록 주의하세요.

### 작은 실행 예제 · 원본 번호와 수업 번호를 구분하기

`zip`은 같은 위치의 값을 짝짓고, `enumerate`는 목록에 순번을 붙입니다. 이 둘로 번호가 어떤 이름을 뜻하는지 확인합니다.

**설명용 가상 데이터입니다. 실제 모델·CIFAR 이미지나 GPU 결과를 계산하지 않습니다.**

In [2]:
classes = ["bottle", "bowl", "can", "cup", "plate"]
original_ids = [9, 10, 16, 28, 61]
original_names = dict(zip(original_ids, classes))
class_ids = {old: new for new, old in enumerate(original_ids)}
print("원본 28번의 이름:", original_names[28])
print("원본 28번 → 수업 번호:", class_ids[28])
print("수업 3번의 이름:", classes[3])

원본 28번의 이름: cup
원본 28번 → 수업 번호: 3
수업 3번의 이름: cup


**확인:** `cup`, `3`, `cup`이 출력됩니다. `classes[0]`은 첫 번째인 `bottle`입니다. 클래스 번호와 이미지 행 번호를 혼동하지 마세요.

## 원본 17번째 셀 · 큰 파일의 지문을 조금씩 계산하는 함수

아래는 **읽기용 원본 코드**입니다.

```python
def digest(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()
```

**입력 → 출력:** 파일 경로 하나를 받아 SHA256 문자열 하나를 돌려주는 `digest` 함수를 정의합니다. 이 셀에서는 함수를 만들 뿐이며 아직 파일을 읽지 않습니다.

- `def digest(path):`는 작업에 이름을 붙입니다. 나중에 `digest(파일경로)`라고 호출할 때 들여쓴 내용이 실행됩니다.
- `hashlib.sha256()`은 지문 계산기를 새로 만듭니다. 함수 안의 `hasher`는 이번 파일 하나를 위해 쓰는 변수입니다.
- `Path(path).open("rb")`는 파일을 **읽기 전용·바이트 모드**로 엽니다. `r`은 읽기, `b`는 바이트라는 뜻입니다.
- `with ... as stream`의 `stream`은 파일에서 내용을 읽는 통로입니다. 구간이 끝나거나 오류가 생기면 파일이 닫힙니다.
- `stream.read(1024 * 1024)`는 한 번에 최대 1 MiB를 읽습니다. 파일 전체를 한꺼번에 메모리에 올리지 않으려는 선택입니다.
- `lambda: ...`는 이 읽기 작업을 짧은 이름 없는 함수로 만든 표현입니다. `iter(작업, b"")`는 결과가 빈 바이트가 될 때까지 같은 작업을 반복합니다.
- 반복문이 읽은 `block`을 `hasher.update(block)`에 차례로 넣습니다. 순서대로 모든 조각을 넣으면 전체 파일의 지문을 얻습니다.
- `return hasher.hexdigest()`가 호출한 곳으로 문자열을 돌려줍니다. 파일 이름이 아니라 **파일 내용**의 지문이며, 없는 경로를 주면 파일을 열 때 오류가 납니다.

## 원본 18번째 셀 · 저장해 둔 이미지·라벨·ID 검사하기

아래는 **읽기용 원본 코드**입니다.

```python
def load_prepared(path):
    path = Path(path)
    manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    classes = manifest["classes"]
    if len(classes) < 2 or len(set(classes)) != len(classes):
        raise ValueError("클래스 목록이 잘못됐습니다.")
    splits, all_ids = {}, set()
    for name in ("train", "validation", "test"):
        record = manifest["splits"][name]
        if record["file"] != f"{name}.npz":
            raise ValueError("데이터 파일 경로가 예상 형식과 다릅니다.")
        file = path / record["file"]
        if digest(file) != record["sha256"]:
            raise ValueError(f"{name} 데이터 체크섬 불일치")
        with np.load(file, allow_pickle=False) as a:
            images, labels, ids = a["images"], a["labels"], a["ids"].tolist()
        if images.dtype != np.uint8 or images.ndim != 4 or images.shape[-1] != 3:
            raise ValueError("images는 uint8 NHWC RGB여야 합니다.")
        if len(images) != len(labels) or len(ids) != len(labels) or len(labels) != record["count"]:
            raise ValueError("이미지·라벨·식별자 개수가 다릅니다.")
        if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or len(labels) == 0 or labels.min() < 0 or labels.max() >= len(classes):
            raise ValueError("라벨 범위가 잘못됐습니다.")
        if len(set(ids)) != len(ids) or all_ids.intersection(ids):
            raise ValueError("분할 간 이미지 ID 중복")
        all_ids.update(ids)
        splits[name] = {"images": images, "labels": labels, "ids": ids}
    identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
    if hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest() != manifest["dataset_sha256"]:
        raise ValueError("데이터 manifest 지문 불일치")
    return splits, manifest
```

**입력 → 출력:** 준비 데이터 폴더를 받아, 검사에 통과한 분할 사전 `splits`와 설명 기록 `manifest`를 돌려줍니다. 함수 정의만 읽어 두고 실제 호출은 뒤에서 합니다.

- `manifest.json`은 파일 목록·클래스·개수 등을 적은 안내장입니다. `.read_text()`로 읽고 `json.loads()`로 Python 사전으로 바꿉니다.
- `classes`에는 클래스 이름 목록이 들어갑니다. 두 종류 이상인지, `set(classes)`로 중복을 제거했을 때 개수가 같은지 검사합니다.
- `splits, all_ids = {}, set()`는 결과용 사전과 이미 읽은 이미지 ID를 모을 집합을 만듭니다. **집합**은 중복 없는 값들을 담습니다.
- `for name in ("train", "validation", "test")`는 세 분할을 순서대로 검사합니다. 파일 이름이 반드시 해당 분할의 `.npz`인지 확인하고, 읽기 전에 파일 지문을 비교합니다.
- `np.load(..., allow_pickle=False)`로 압축 배열 묶음을 엽니다. 여기서는 Python 객체를 복원하는 pickle 기능을 허용하지 않고 이미지·숫자·문자 배열만 읽습니다.
- `images`, `labels`, `ids`는 각각 이미지 숫자 배열, 정답 번호 배열, 원본 위치를 나타내는 ID 목록입니다. 한 사진의 세 정보가 같은 위치에 있어야 합니다.
- `uint8`은 픽셀을 0~255 정수로 저장하는 자료형입니다. `ndim == 4`와 마지막 길이 `3`은 `(사진 수, 높이, 너비, RGB)` 구조인지 확인합니다. 이 검사 자체가 높이·너비를 꼭 32로 제한하지는 않습니다.
- `len(...)` 검사들은 이미지·라벨·ID·기록된 개수가 서로 같은지 확인합니다. 라벨은 일차원 정수 배열이고, 값은 `0`부터 `클래스 수 - 1`까지여야 합니다.
- `set(ids)`로 분할 안의 ID 중복을 검사하고, `all_ids.intersection(ids)`로 앞서 읽은 다른 분할과 겹치는 ID를 찾습니다. 통과하면 `.update(ids)`로 ID를 모읍니다.
- `splits[name] = {...}`는 검사한 데이터를 분할 이름으로 저장합니다. `splits["train"]["images"]`처럼 두 번 찾아 학습 이미지를 꺼낼 수 있습니다.
- 마지막의 `identity`는 클래스·파일 기록·seed·전처리 설정을 모은 사전입니다. `sort_keys=True`로 항목 순서를 정해 지문을 계산한 뒤 기록과 비교합니다.
- `return splits, manifest`는 두 값을 돌려줍니다. 이 함수의 ID 검사는 분할 중복을 막는 한 방법이며, 실제 픽셀 내용이 같은지 확인하는 검사는 저장 함수에 따로 있습니다.

## 원본 20번째 셀 · 같은 조건의 준비 데이터인지 판단하기

아래는 **읽기용 원본 코드**입니다.

```python
selection = {"train_per_class": TRAIN_PER_CLASS, "val_per_class": VALIDATION_PER_CLASS,
             "test_per_class": TEST_PER_CLASS}
REUSE_PREPARED = (DATA_PATH / "manifest.json").is_file()
if REUSE_PREPARED:
    splits, manifest = load_prepared(DATA_PATH)
    source = manifest.get("source", {})
    if (manifest["dataset_name"] != "cifar100_food_containers" or manifest["seed"] != SEED
            or manifest["classes"] != CLASSES or source.get("selection") != selection
            or source.get("revision") != DATASET_REVISION or source.get("source_sha256") != SOURCE_FILES):
        raise ValueError("다른 조건의 데이터가 있습니다. 새 DATA_PATH를 지정하세요.")
    print("기존 데이터의 SHA256과 실험 조건을 확인했습니다.")
elif DATA_PATH.exists() and any(DATA_PATH.iterdir()):
    raise FileExistsError("완성되지 않은 데이터 폴더가 있습니다. 내용을 확인하거나 새 DATA_PATH를 지정하세요.")
```

**입력 → 출력:** `DATA_PATH`에 이미 준비한 파일이 있는지 확인하고 `REUSE_PREPARED`를 정합니다. 같은 조건의 데이터만 재사용합니다.

- `selection`은 클래스마다 몇 장을 선택했는지 기록한 사전입니다. 숫자가 같아야 같은 조건의 실험이라고 볼 수 있습니다.
- `(DATA_PATH / "manifest.json").is_file()`의 결과는 `True` 또는 `False`입니다. 이를 `REUSE_PREPARED`에 담아 뒤의 여러 셀에서 공통으로 사용합니다.
- 기록 파일이 있으면 `load_prepared(DATA_PATH)`를 호출합니다. 앞서 정의한 무결성 검사를 실제로 실행하고 `splits`, `manifest` 두 값을 받습니다.
- `manifest.get("source", {})`는 `source` 항목이 있으면 그 값을, 없으면 빈 사전을 가져옵니다. `.get(...)`은 누락된 항목을 검사할 때 유용합니다.
- 긴 `if (...)`는 데이터 이름, seed, 클래스 순서, 분할 수, revision, 원본 지문 중 **하나라도 다르면** 중단합니다. `or`는 ‘또는’입니다.
- 같은 조건이면 재사용 확인 문구를 출력합니다. 다운로드를 생략하더라도 실제 파일과 조건은 다시 확인한 셈입니다.
- `elif`는 앞의 조건이 거짓일 때 다음 경우를 검사합니다. 폴더가 존재하고 `any(DATA_PATH.iterdir())`가 참이면, 기록 없이 파일만 남은 상태로 판단해 멈춥니다.
- 다른 실험을 하려면 `DATA_PATH`를 새 폴더로 바꿉니다. 오류를 무시하거나 기존 기록만 지워 덮어쓰면 어떤 조건의 데이터인지 알기 어려워집니다.

## 원본 22번째 셀 · 원본 파일을 임시 이름으로 받고 확인하기

아래는 **읽기용 원본 코드**입니다.

```python
CACHE.mkdir(parents=True, exist_ok=True)
source_paths = {}
if not REUSE_PREPARED:
    for split_name in ("train", "test"):
        filename = f"{split_name}-00000-of-00001.parquet"
        target = CACHE / filename
        if not target.is_file():
            partial = target.with_suffix(".part")
            url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/cifar100/{filename}"
            print("Download:", filename)
            try:
                with urllib.request.urlopen(url, timeout=120) as response, partial.open("wb") as stream:
                    shutil.copyfileobj(response, stream)
                if digest(partial) != SOURCE_FILES[split_name]:
                    raise ValueError("다운로드한 파일의 SHA256이 다릅니다.")
                partial.replace(target)
            finally:
                partial.unlink(missing_ok=True)
        if digest(target) != SOURCE_FILES[split_name]:
            raise ValueError(f"캐시 SHA256 불일치: {target}")
        source_paths[split_name] = target
        print("Verified:", filename)
else:
    print("검증한 준비 데이터를 재사용하므로 원본 다운로드를 생략합니다.")
```

**입력 → 출력:** 처음 준비하는 경우 원본 학습·테스트 Parquet를 내려받아 검증하고, `source_paths`에 두 경로를 저장합니다.

- `CACHE.mkdir(parents=True, exist_ok=True)`는 상위 폴더까지 만들고, 이미 있어도 오류를 내지 않습니다. 재사용하는 경우에도 이 폴더 준비는 실행됩니다.
- `if not REUSE_PREPARED`는 준비 데이터를 재사용하지 않을 때만 원본을 받게 합니다. 이미 검증한 데이터가 있으면 맨 아래 안내문만 출력합니다.
- 반복문은 `train`, `test` 두 이름으로 파일명을 만듭니다. `f"{split_name}-..."`의 중괄호 자리에 현재 이름이 들어갑니다.
- `target`은 완성될 파일, `.with_suffix(".part")`는 다운로드 중 사용할 임시 파일 경로입니다. 미완성 파일을 정상 파일로 착각하지 않도록 구분합니다.
- 인터넷 주소에는 `DATASET_ID`, 고정 revision, 파일명이 함께 들어갑니다. 모델을 받았던 주소와 **데이터를 받는 주소는 별개**입니다.
- `urllib.request.urlopen(..., timeout=120)`은 주소를 열고, `partial.open("wb")`는 임시 파일을 바이트 쓰기 모드로 엽니다. `shutil.copyfileobj`가 받은 내용을 파일에 옮깁니다.
- `try` 안에서 다운로드와 지문 확인을 하고, 통과한 경우만 `partial.replace(target)`으로 완성 이름을 붙입니다.
- `finally`는 성공·실패와 관계없이 실행됩니다. 남은 임시 파일은 `unlink(missing_ok=True)`로 지우며, 이미 없어도 괜찮습니다.
- 이미 캐시에 있던 파일도 `digest(target)`로 검사합니다. 이름만 같다고 믿고 넘어가지 않습니다. 통과하면 `source_paths[split_name]`에 경로를 남깁니다.
- ‘Verified’가 출력됐다는 것은 이 파일의 지문 확인을 통과했다는 뜻입니다. 이미지 선택과 모델 추론은 아직 시작하지 않았습니다.

## 원본 24번째 셀 · Parquet 표에서 필요한 열만 읽기

아래는 **읽기용 원본 코드**입니다.

```python
if not REUSE_PREPARED:
    train_table = pq.read_table(source_paths["train"], columns=["img", "fine_label"])
    test_table = pq.read_table(source_paths["test"], columns=["img", "fine_label"])
    train_labels = train_table["fine_label"].to_numpy()
    test_labels = test_table["fine_label"].to_numpy()
    assert len(train_labels) == 50000 and len(test_labels) == 10000
    print(train_table.schema)
    print("Original split sizes:", len(train_labels), len(test_labels))
```

**입력 → 출력:** 원본 파일 두 개에서 이미지 열과 정답 열을 읽고, 원본 분할의 개수를 확인합니다.

- 이 셀도 `not REUSE_PREPARED`일 때만 실행합니다. 이미 준비된 배열을 읽은 경우 원본 표를 다시 메모리에 올릴 필요가 없습니다.
- `pq.read_table(source_paths["train"], ...)`는 앞서 확인한 학습 파일을 읽습니다. 테스트 파일은 별도의 `test_table`로 읽습니다.
- `columns=["img", "fine_label"]`은 필요한 두 열만 지정합니다. `img`에는 이미지 정보, `fine_label`에는 원본의 100종류 정답 번호가 있습니다.
- `train_table["fine_label"]`은 표에서 그 열만 꺼냅니다. `.to_numpy()`는 다음 NumPy 연산에 사용할 숫자 배열로 바꿉니다.
- `train_labels`와 `test_labels`는 모든 원본 사진의 정답 번호를 순서대로 갖고 있습니다. 아직 수업의 다섯 종류만 추린 것이 아닙니다.
- `assert len(...) == 50000 and ... == 10000`은 두 개수 조건이 모두 맞는지 확인합니다. 맞지 않으면 `AssertionError`로 멈춥니다.
- `train_table.schema`는 표의 열 이름과 자료형을 보여줍니다. `Original split sizes`는 **선택 전 원본 개수**라서 수업용 500·100·200과 다르게 나오는 것이 정상입니다.
- `source_paths`가 없다는 오류가 나면 앞의 다운로드 셀부터 순서대로 실행했는지 살펴보세요. 셀을 따로 실행하면 필요한 변수가 아직 없을 수 있습니다.

## 원본 26번째 셀 · 학습·검증·테스트가 겹치지 않게 고르기

아래는 **읽기용 원본 코드**입니다.

```python
if not REUSE_PREPARED:
    if min(TRAIN_PER_CLASS, VALIDATION_PER_CLASS, TEST_PER_CLASS) < 1:
        raise ValueError("클래스별 선택 수는 양수여야 합니다.")
    rng = np.random.default_rng(SEED)
    train_indices, validation_indices = [], []
    for label in FINE_LABEL_IDS:
        candidates = np.flatnonzero(train_labels == label)
        if TRAIN_PER_CLASS + VALIDATION_PER_CLASS > len(candidates):
            raise ValueError("학습·검증 요청 수가 원본 클래스 수를 넘습니다.")
        shuffled = rng.permutation(candidates)
        train_indices.extend(shuffled[:TRAIN_PER_CLASS])
        validation_indices.extend(shuffled[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VALIDATION_PER_CLASS])
    rng = np.random.default_rng(SEED + 1)
    if TEST_PER_CLASS > 100:
        raise ValueError("원본 테스트에는 클래스마다 100장이 있습니다.")
    test_indices = np.concatenate([rng.permutation(np.flatnonzero(test_labels == label))[:TEST_PER_CLASS]
                                   for label in FINE_LABEL_IDS])
    assert not set(train_indices).intersection(validation_indices)
    print("Selected:", len(train_indices), len(validation_indices), len(test_indices))
```

**입력 → 출력:** 원본 정답 배열과 클래스별 선택 수를 바탕으로 이미지의 **행 번호** 세 묶음을 고릅니다. 아직 사진 픽셀을 옮기지는 않습니다.

- `min(...) < 1`은 세 선택 수 중 하나라도 0 이하인지 검사합니다. 학습·검증·테스트에는 모두 이미지가 있어야 합니다.
- `np.random.default_rng(SEED)`는 일정한 seed에서 출발하는 무작위 생성기를 만듭니다. 같은 조건과 실행 순서에서 같은 선택을 다시 얻도록 합니다.
- `train_labels == label`은 각 행이 현재 클래스인지 비교합니다. `np.flatnonzero(...)`는 참인 위치, 즉 **후보 행 번호**를 모읍니다.
- 클래스 하나에서 요청한 학습 수와 검증 수의 합이 후보 수를 넘으면 중단합니다. 충분한 후보가 있어야 겹치지 않게 뽑을 수 있습니다.
- `rng.permutation(candidates)`는 후보 순서를 섞습니다. `shuffled[:100]`은 앞 100개, `shuffled[100:120]`은 100번 위치부터 120번 직전까지입니다. Python 위치는 0부터 시작합니다.
- `.extend(...)`는 고른 행 번호들을 기존 목록 뒤에 하나씩 덧붙입니다. 다섯 클래스를 처리하면 학습 500개, 검증 100개의 행 번호가 모입니다.
- 테스트는 `SEED + 1`로 새 생성기를 만들고 **원본 테스트 파일**에서 고릅니다. 원본 학습 파일에서 테스트를 다시 떼는 코드가 아닙니다.
- 테스트에는 클래스별 원본이 100장이라 요청 수를 검사합니다. 목록 안의 `for label in ...`은 클래스별 선택을 짧게 쓴 반복문이고, `np.concatenate`는 결과를 한 배열로 이어 붙입니다.
- 마지막 `intersection` 검사에서 학습·검증 행 번호의 교집합이 비어 있어야 합니다. 출력 `Selected: 500 100 200`은 기본 설정에서 예상하는 선택 개수입니다.
- 검증 데이터는 학습 도중 어느 상태가 좋은지 고를 때, 테스트 데이터는 선택이 끝난 뒤 평가할 때 씁니다. 테스트 결과를 보며 계속 설정을 바꾸면 독립적인 최종 평가 역할이 약해집니다.

### 작은 실행 예제 · 섞은 뒤 서로 다른 구간에서 고르기

작은 ID 목록을 섞은 뒤 앞부분은 학습, 그다음은 검증으로 나눕니다. Python의 `random`을 쓰므로 원본 NumPy가 뽑는 실제 행 번호와 같지는 않습니다. 분리 원리만 확인합니다.

**설명용 가상 데이터입니다. 실제 모델·CIFAR 이미지나 GPU 결과를 계산하지 않습니다.**

In [3]:
import random

ids = list(range(10))
random.Random(42).shuffle(ids)
train_ids = ids[:6]
validation_ids = ids[6:8]
test_ids = ["test/0", "test/1"]  # 별도 원본에서 왔다고 가정합니다.
print("학습:", train_ids)
print("검증:", validation_ids)
print("학습·검증이 겹치는가:", bool(set(train_ids) & set(validation_ids)))
print("테스트:", test_ids)
again = list(range(10))
random.Random(42).shuffle(again)
print("같은 seed로 같은 순서를 얻었는가:", ids == again)

학습: [7, 3, 2, 8, 5, 6]
검증: [9, 4]
학습·검증이 겹치는가: False
테스트: ['test/0', 'test/1']
같은 seed로 같은 순서를 얻었는가: True


**확인:** 학습은 `[7, 3, 2, 8, 5, 6]`, 검증은 `[9, 4]`이며 겹침은 `False`입니다. seed를 바꾸면 어떤 ID를 고르는지 달라질 수 있지만, 서로 겹치지 않는 구간으로 나누는 원칙은 같습니다.

## 원본 28번째 셀 · 이미지 바이트를 RGB 숫자 배열로 바꾸기

아래는 **읽기용 원본 코드**입니다.

```python
if not REUSE_PREPARED:
    label_map = {old: new for new, old in enumerate(FINE_LABEL_IDS)}
    splits = {}
    for name, table, labels, indices, origin in [
        ("train", train_table, train_labels, train_indices, "train"),
        ("validation", train_table, train_labels, validation_indices, "train"),
        ("test", test_table, test_labels, test_indices, "test"),
    ]:
        pixels = []
        for row in table.take([int(index) for index in indices])["img"].to_pylist():
            with Image.open(io.BytesIO(row["bytes"])) as image:
                pixels.append(np.asarray(image.convert("RGB")))
        splits[name] = {"images": np.stack(pixels),
                        "labels": np.asarray([label_map[int(labels[index])] for index in indices], dtype=np.int64),
                        "ids": [f"cifar100/{origin}/{index:05d}" for index in indices]}
    del train_table, test_table
    print("Train pixels:", splits["train"]["images"].shape, splits["train"]["images"].dtype)
```

**입력 → 출력:** 선택한 행의 이미지를 읽어 RGB 픽셀 배열, 새 라벨 `0~4`, 원본 ID를 만든 뒤 `splits`에 넣습니다.

- `{old: new for new, old in enumerate(FINE_LABEL_IDS)}`는 원본 번호를 수업 번호로 바꾸는 사전입니다. `enumerate`는 값에 `0`부터 시작하는 순번을 붙입니다.
- 예를 들어 원본 번호 `28`은 `label_map[28] == 3`으로 바뀝니다. 모델이 예측할 다섯 라벨을 빠짐없이 연속 번호로 맞추려는 작업입니다.
- 바깥 반복문의 한 항목에는 `name, table, labels, indices, origin` 다섯 값이 들어 있습니다. 한 줄에서 이를 나눠 받습니다. 검증도 원본 **train** 표를 사용한다는 점을 보세요.
- `table.take([...])`는 고른 행 번호의 행만 꺼냅니다. `int(index)`는 NumPy 정수 등을 기본 Python 정수로 바꿔 전달합니다.
- 이미지 열을 `.to_pylist()`로 바꿔 하나씩 읽습니다. `row["bytes"]`에는 압축된 이미지 내용이 들어 있습니다.
- `io.BytesIO(...)`는 메모리의 바이트를 파일처럼 제공하고, `Image.open(...)`이 이미지를 읽습니다. `.convert("RGB")`로 빨강·초록·파랑 세 채널을 맞춥니다.
- `np.asarray(...)`는 이미지 픽셀을 숫자 배열로 바꾸며 `.append(...)`는 사진 하나를 `pixels` 목록 뒤에 넣습니다. `np.stack(pixels)`는 여러 사진을 하나의 배열로 쌓습니다.
- 라벨 배열은 대응하는 원본 정답을 `label_map`으로 바꾸고 `np.int64` 정수형으로 저장합니다. **이미지와 라벨의 순서가 같아야** 올바른 정답을 붙일 수 있습니다.
- ID의 `{index:05d}`는 행 번호를 다섯 자리로 보여주는 문법입니다. `cifar100/train/00028` 같은 ID는 클래스 번호가 아니라 원본 파일의 위치를 나타냅니다.
- `del train_table, test_table`은 더 이상 필요 없는 표의 변수 참조를 없애 메모리를 정리하는 데 도움을 줍니다. 저장된 `splits`는 유지됩니다.
- 기본 설정의 학습 배열 출력은 `(500, 32, 32, 3) uint8`입니다. **500장 × 높이 32 × 너비 32 × 색 3개**이며, 224×224 전처리는 뒤의 GPU 노트북에서 합니다.

### 작은 실행 예제 · RGB 이미지를 숫자 목록으로 보기

1장짜리 2×2 이미지를 중첩 목록으로 표현합니다. 가장 안쪽 세 숫자가 한 픽셀의 빨강·초록·파랑 값입니다.

**설명용 가상 데이터입니다. 실제 모델·CIFAR 이미지나 GPU 결과를 계산하지 않습니다.**

In [4]:
image = [
    [[255, 0, 0], [0, 255, 0]],
    [[0, 0, 255], [255, 255, 255]],
]
images = [image]
shape = (len(images), len(images[0]), len(images[0][0]), len(images[0][0][0]))
print("(사진 수, 높이, 너비, 채널):", shape)
print("첫 사진의 왼쪽 위 픽셀:", images[0][0][0])

(사진 수, 높이, 너비, 채널): (1, 2, 2, 3)
첫 사진의 왼쪽 위 픽셀: [255, 0, 0]


**확인:** 구조는 `(1, 2, 2, 3)`, 왼쪽 위 픽셀은 `[255, 0, 0]`입니다. 원본의 `(500, 32, 32, 3)`도 같은 순서로 읽습니다. 이 예제는 숫자 구조만 살펴보며 실제 사진을 그리지는 않습니다.

## 원본 30번째 셀 · 전처리 약속을 기록할 사전 만들기

아래는 **읽기용 원본 코드**입니다.

```python
PREPROCESS = {"size": 224, "mean": [0.5] * 3, "std": [0.5] * 3,
              "resize": "bilinear", "layout": "NHWC"}
```

**입력 → 출력:** 이후 모델 입력을 만들 때 사용할 전처리 조건을 `PREPROCESS` 사전에 담습니다. 이 셀은 이미지 숫자를 바꾸지 않습니다.

- `size: 224`는 이후 모델에 들어갈 이미지의 목표 크기입니다. 앞에서 저장할 픽셀은 여전히 32×32입니다.
- `mean`과 `std`는 정규화에 사용할 값입니다. 정규화는 픽셀 숫자를 모델이 기대하는 범위로 바꾸는 계산입니다.
- `[0.5] * 3`은 `[0.5, 0.5, 0.5]`라는 목록을 만듭니다. 0.5를 3으로 곱해 숫자 1.5 하나를 만드는 것이 아닙니다.
- 세 값은 RGB 세 채널에 대응합니다. 이 딕셔너리는 그 약속을 기록하며 실제 계산은 다음 실습의 전처리 도구가 수행합니다.
- `resize: "bilinear"`는 크기를 바꿀 때 주변 픽셀을 이용하는 보간 방식을 기록합니다. 작은 이미지를 키워도 원래 없던 세부 정보가 복구되지는 않습니다.
- `layout: "NHWC"`는 저장 배열의 축 순서입니다. N은 사진 수, H는 높이, W는 너비, C는 색 채널 수입니다.
- 사전의 `"size"` 같은 문자열은 항목 이름, `224` 같은 값은 설정 내용입니다. `PREPROCESS["size"]`로 값을 꺼낼 수 있습니다.
- 이 설정도 데이터 지문에 포함됩니다. 값을 바꿨다고 앞서 만든 픽셀이 자동으로 변환되지는 않으므로 기록과 실제 처리 코드를 함께 이해해야 합니다.

## 원본 31번째 셀 · 분할 파일과 설명 기록 저장하기

아래는 **읽기용 원본 코드**입니다.

```python
def save_prepared(output, splits, classes, source, seed):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    # Exact duplicate pixels across splits are excluded before this point.
    seen = {}
    for name, split in splits.items():
        for image in split["images"]:
            key = hashlib.sha256(image.tobytes()).hexdigest()
            if key in seen and seen[key] != name:
                raise ValueError(f"분할 간 중복 이미지: {seen[key]} / {name}")
            seen[key] = name
    records = {}
    for name, split in splits.items():
        path = output / f"{name}.npz"
        with path.with_suffix(".tmp").open("wb") as stream:
            np.savez_compressed(stream, images=split["images"], labels=split["labels"], ids=np.asarray(split["ids"], dtype=str))
        path.with_suffix(".tmp").replace(path)
        records[name] = {"file": path.name, "sha256": digest(path), "count": len(split["labels"]),
                         "per_class": {c: int(np.sum(split["labels"] == i)) for i, c in enumerate(classes)}}
    identity = {"classes": classes, "splits": records, "seed": seed, "preprocess": PREPROCESS}
    manifest = {"schema_version": 1, "dataset_name": source["name"], **identity, "source": source,
                "dataset_sha256": hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()}
    (output / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return manifest
```

**입력 → 출력:** 준비한 `splits`, 클래스 목록, 출처, seed를 받아 `.npz` 세 개와 `manifest.json`을 저장하는 함수를 정의합니다.

- `output = Path(output)`로 저장 위치를 경로로 바꾸고 폴더를 만듭니다. 함수 안에서 쓰는 `output`, `records` 같은 이름은 이 작업을 위한 변수입니다.
- 첫 반복문은 각 이미지의 `image.tobytes()`로 픽셀 바이트를 얻어 지문을 계산합니다. 파일 이름이 아니라 **픽셀 내용이 같은지** 확인하려는 검사입니다.
- `seen`은 ‘이미지 지문 → 처음 본 분할 이름’ 사전입니다. 같은 지문이 다른 분할에서 나오면 중단합니다. 이 코드가 검사하는 것은 정확히 같은 픽셀이며, 비슷한 사진까지 찾는 것은 아닙니다.
- 주석에 중복 제외가 언급돼 있지만 실제 동작은 중복 사진을 자동 삭제하는 것이 아니라 **오류로 중단하는 것**입니다. 같은 분할 안의 중복도 이 조건만으로는 막지 않습니다.
- 다음 반복문에서 `train.npz`처럼 분할별 파일 이름을 정합니다. `.tmp` 임시 파일을 먼저 열어 완성되기 전 상태를 구분합니다.
- `np.savez_compressed`는 `images`, `labels`, `ids` 세 배열을 이름이 붙은 압축 묶음으로 저장합니다. `ids`는 문자열 배열로 바꿉니다.
- 쓰기를 마치면 `.replace(path)`로 정식 파일 이름으로 바꿉니다. 이 함수는 **전달받은 저장 위치에 파일을 씁니다.** 앞의 재사용 검사와 함께 사용해야 합니다.
- `records[name]`에는 파일 이름·지문·이미지 수·클래스별 개수를 남깁니다. `np.sum(labels == i)`는 해당 클래스인 항목 수를 셉니다.
- `identity`는 클래스, 분할 기록, seed, 전처리 설정입니다. `**identity`는 이 사전의 항목들을 다른 사전 안에 펼쳐 넣습니다.
- `dataset_sha256`는 정해진 순서의 JSON으로 바꾼 `identity`에서 계산합니다. 출처 전체가 아니라 이 네 항목을 기준으로 하는 지문입니다. 출처 조건은 앞의 재사용 셀에서 별도로 비교합니다.
- `ensure_ascii=False`는 한글을 그대로 읽을 수 있게 저장하고 `indent=2`는 들여쓰기를 넣습니다. 마지막 `return manifest`는 저장한 설명 기록을 호출한 셀에서도 쓰게 돌려줍니다.
- 이 셀에서는 함수를 정의할 뿐 저장하지 않습니다. 바로 다음 셀의 `save_prepared(...)` 호출이 실제 저장을 시작합니다.

## 원본 32번째 셀 · 저장한 데이터를 다시 열어 확인하기

아래는 **읽기용 원본 코드**입니다.

```python
if not REUSE_PREPARED:
    source = {"name": "cifar100_food_containers", "dataset_id": DATASET_ID,
              "revision": DATASET_REVISION, "source_sha256": SOURCE_FILES,
              "original_url": "https://www.cs.toronto.edu/~kriz/cifar.html",
              "original_resolution": [32, 32], "selection": selection}
    manifest = save_prepared(DATA_PATH, splits, CLASSES, source, SEED)
splits, manifest = load_prepared(DATA_PATH)
classes = manifest["classes"]
for name, split in splits.items():
    print(name, dict(zip(classes, np.bincount(split["labels"], minlength=len(classes)).tolist())))
print("Dataset fingerprint:", manifest["dataset_sha256"])
```

**입력 → 출력:** 새로 만든 데이터라면 파일을 저장하고, 재사용 여부와 관계없이 파일을 다시 검증해 클래스별 개수와 데이터 지문을 출력합니다.

- `if not REUSE_PREPARED` 구간은 새 데이터를 만드는 경우에만 실행됩니다. `source`에는 데이터 이름, Hub 주소, revision, 원본 파일 지문과 선택 개수가 들어갑니다.
- `original_resolution: [32, 32]`는 원본의 픽셀 크기 기록입니다. `selection`은 앞서 정한 클래스별 학습·검증·테스트 개수입니다.
- `save_prepared(DATA_PATH, splits, CLASSES, source, SEED)`에서 다섯 인자의 순서는 함수 정의의 `output, splits, classes, source, seed`와 대응합니다.
- `splits, manifest = load_prepared(DATA_PATH)`는 `if` 바깥에 있습니다. 따라서 처음 저장한 경우와 기존 파일을 재사용한 경우 **모두** 디스크의 파일을 다시 검사합니다.
- `classes = manifest["classes"]`는 저장된 클래스 순서를 가져옵니다. 대문자 `CLASSES`는 처음 지정한 조건, 소문자 `classes`는 기록에서 다시 읽은 값입니다.
- `splits.items()`는 이름과 분할 데이터를 짝으로 꺼냅니다. `np.bincount(...)`는 `0`, `1`, `2`처럼 정수 라벨별 개수를 셉니다.
- `minlength=len(classes)`는 모든 클래스가 출력에 나타나도록 최소 길이를 맞춥니다. `.tolist()`로 일반 목록으로 바꾸고 `dict(zip(...))`으로 이름과 개수를 연결합니다.
- 기본 설정에서는 각 클래스가 학습 100장, 검증 20장, 테스트 40장입니다. 다섯 클래스의 합계가 각각 500·100·200인지 함께 확인하면 좋습니다.
- 마지막 데이터 지문은 ‘이 분할과 설정의 기록’을 비교할 때 쓰는 값입니다. 모델의 점수나 정답률을 나타내는 숫자가 아닙니다.

### 작은 실행 예제 · JSON은 기록을 주고받는 글자 형식입니다

간단한 기록을 JSON 문자열로 바꿨다가 다시 사전으로 읽습니다. 파일에는 저장하지 않습니다.

**설명용 가상 데이터입니다. 실제 모델·CIFAR 이미지나 GPU 결과를 계산하지 않습니다.**

In [5]:
import json

record = {"classes": ["cup", "bowl"], "seed": 42, "counts": {"train": 6, "validation": 2, "test": 2}}
text = json.dumps(record, ensure_ascii=False, indent=2)
restored = json.loads(text)
print("JSON 안에 seed 항목이 있는가:", '"seed"' in text)
print("다시 읽은 클래스:", restored["classes"])
print("기록의 합계:", sum(restored["counts"].values()))
print("원래 사전과 같은가:", restored == record)

JSON 안에 seed 항목이 있는가: True
다시 읽은 클래스: ['cup', 'bowl']
기록의 합계: 10
원래 사전과 같은가: True


**확인:** `True`, `['cup', 'bowl']`, `10`, `True`가 출력됩니다. 기록 안의 개수와 실제 파일 내용을 대조하는 일은 원본의 검사 함수가 따로 수행합니다.

## 원본 34번째 셀 · 학습 이미지 15장을 눈으로 확인하기

아래는 **읽기용 원본 코드**입니다.

```python
fig, axes = plt.subplots(len(classes), 3, figsize=(8, 2 * len(classes)), squeeze=False)
for label, name in enumerate(classes):
    indices = np.flatnonzero(splits["train"]["labels"] == label)[:3]
    for column, index in enumerate(indices):
        axes[label, column].imshow(splits["train"]["images"][index], interpolation="nearest")
        axes[label, column].set_title(f"{name} · sample {column + 1}")
        axes[label, column].axis("off")
fig.suptitle("Training images · CIFAR-100 food containers · native 32 × 32", y=1.01)
fig.tight_layout()
plt.show()
```

**입력 → 출력:** 다섯 클래스의 학습 이미지에서 각각 처음 세 장을 골라 5행×3열 그림으로 보여줍니다. 여기서 ‘처음’은 앞서 무작위로 고른 배열의 현재 순서입니다.

- `plt.subplots(len(classes), 3, ...)`는 다섯 행과 세 열의 그림 자리를 만듭니다. `fig`는 전체 그림, `axes`는 각 작은 그림 자리입니다.
- `figsize=(8, 2 * len(classes))`는 그림 전체의 가로·세로 크기입니다. 클래스가 늘면 세로도 길어지도록 했습니다.
- `squeeze=False`는 그림 자리들을 늘 이차원 구조로 받도록 합니다. 그래서 `axes[행, 열]`처럼 일관되게 찾을 수 있습니다.
- `enumerate(classes)`로 클래스 순번과 이름을 함께 얻습니다. `splits["train"]["labels"] == label`로 지금 볼 클래스의 사진을 찾습니다.
- `np.flatnonzero(... )[:3]`은 그 사진들의 위치 중 앞 세 개를 고릅니다. 모든 학습 이미지를 보여주는 셀은 아닙니다.
- 안쪽 반복문은 열 번호와 이미지 위치를 받아 `imshow(...)`에 해당 픽셀 배열을 넘깁니다. `interpolation="nearest"`는 원래 픽셀의 경계를 비교적 그대로 보여줍니다.
- `set_title`은 물체 이름과 샘플 번호, `axis("off")`는 사진 주변 좌표 눈금 숨김을 뜻합니다. `column + 1`은 사람에게 번호를 1부터 보여주기 위한 계산입니다.
- `fig.suptitle`은 전체 제목, `tight_layout`은 그림 사이 간격 조정, `plt.show()`는 완성한 그림 표시입니다.
- 출력은 모델의 추론 결과가 아니라 **정답 라벨이 있는 입력 이미지**입니다. 컵·그릇의 윤곽과 배경을 보면서 어떤 사진이 어려울지 먼저 예상해 보세요.

## 준비가 끝났을 때 무엇이 남을까요?

아래는 **원본 00번을 기본 설정으로 정상 실행했을 때**의 파일 구성입니다. 이 읽기용 해설 노트북이 만든 파일 목록은 아닙니다.

| 위치 | 내용 |
|---|---|
| `hf_colab_gpu/models/deit-tiny/` | 모델 설정, 전처리 설정, 가중치 세 파일과 다운로드 기록 |
| `data/prepared/train.npz` | 학습 이미지 500장과 라벨·ID |
| `data/prepared/validation.npz` | 검증 이미지 100장과 라벨·ID |
| `data/prepared/test.npz` | 테스트 이미지 200장과 라벨·ID |
| `data/prepared/manifest.json` | 출처, 클래스 순서, 분할 개수, 파일 지문, 전처리 조건 |

32×32 이미지를 준비했지만 모델이 사용할 입력 크기는 224×224입니다. **준비 단계에서 저장하는 배열**과 **추론 직전에 모델에 넣는 배열**을 구분하면 다음 노트북을 읽기 쉬워집니다.

## 다시 찾기 좋은 용어

| 용어 | 이 실습에서의 뜻 |
|---|---|
| 라이브러리 | 자주 쓰는 작업을 미리 구현해 둔 도구 모음 |
| 커널 | 노트북의 Python 코드를 실행하는 프로그램 |
| CLI | 글자로 명령을 입력해 사용하는 도구 |
| revision | 모델이나 데이터의 특정 버전을 가리키는 값 |
| 라벨 | 사진의 정답 종류를 나타내는 번호 |
| ID | 개별 사진이 원본 어디에서 왔는지 구분하는 값 |
| split | 학습·검증·테스트처럼 역할에 따라 나눈 데이터 묶음 |
| seed | 무작위 선택을 재현하기 위한 시작값 |
| SHA256 | 파일이나 바이트 내용으로 계산하는 비교용 지문 |
| NPZ | 이름 붙인 NumPy 배열 여러 개를 묶어 저장하는 형식 |
| manifest | 데이터의 구성과 출처를 설명하는 기록 |

## 이해했는지 확인하기

1. `MODEL_ID`와 `MODEL_DIR`는 각각 무엇을 가리키나요?
2. 다운로드를 마친 뒤 파일 지문을 다시 확인하는 이유는 무엇인가요?
3. 클래스 번호 `3`과 `cifar100/train/00003`은 어떻게 다른가요?
4. 기본 설정에서 학습 이미지가 500장이 되는 계산을 설명해 보세요.
5. 검증 이미지를 학습에도 넣으면 어느 선택 과정에 영향을 줄까요?
6. `PREPROCESS["size"]`가 224인데 저장 배열이 32×32인 이유는 무엇인가요?
7. 폴더에 다른 seed로 만든 데이터가 있을 때 왜 새 폴더를 쓰라고 할까요?

한 질문씩 AI에게 “원본의 어느 변수와 코드 줄이 근거인지 함께 설명해 줘”라고 요청해 보세요. 숫자나 이름을 바꿔 생각해 본 뒤, 답이 실제 코드와 맞는지 직접 확인하는 연습이 됩니다.

## 다음 단계

- 실제 파일 준비: [00 · 모델과 이미지 준비](../notebooks/00_hf_download_and_data.ipynb)
- GPU 연결과 파일 전송: [실행 안내](../README.md)
- 다음 실습: [01 · GPU 추론](../notebooks/01_gpu_inference.ipynb)
- 다음 코드 해설: [01 · GPU 추론 해설](01_gpu_inference_explained.ipynb)

01번은 Colab GPU에서 실행합니다. Codespaces에서 파일을 열었다고 GPU가 자동으로 연결되지는 않습니다. 원본과 해설의 역할을 구분해 이어가세요.